# `convert_synthetic_robust.py` — JSON → CSV Converter for Synthetic Reviews

## Purpose
Converts **LLM-generated synthetic review JSON files** into the standardised CSV format used by the rest of the data pipeline.

## Pipeline
```
Input : data/augmented/*.json   (raw LLM output — may be malformed)
Output: data/augmented/*.csv    (one CSV per JSON, same base filename)
```

## Why "robust"?
LLMs sometimes generate JSON that has **trailing commas**, **missing brackets**, or other structural errors. Instead of using `json.load()` which would fail on the first error, this script uses a **non-nested regex strategy** to extract individual `{ ... }` review objects from the raw text, parsing them one at a time. Invalid objects are silently skipped.

## Output columns
| Column | Description |
|--------|-------------|
| `data` | The review text |
| `<aspects...>` | One column per sentiment aspect (e.g. colour, smell, price) |
| `text_clean` | Cleaned version of `data` |
| `signature` | Compact deduplication key (e.g. `colour:positive\|smell:na\|...`) |

In [ ]:
import os
import json
import argparse
import logging
import pandas as pd
import re
import yaml
import emoji          # Removes emoji characters during text cleaning
from cleantext import clean  # cleantext library applies standard text normalisation
from pathlib import Path

## Utility Functions
These are small helpers that are used by the main processing logic.

In [ ]:
def load_cfg(project_dir: str) -> tuple:
    """Load configs/config.yaml and return (project_dir_path, cfg dict)."""
    root     = Path(project_dir).resolve()
    cfg_path = root / 'configs' / 'config.yaml'
    with open(cfg_path, 'r', encoding='utf-8') as f:
        cfg = yaml.safe_load(f)  # safe_load prevents arbitrary Python execution from the YAML
    return root, cfg


def setup_logger(project_dir: Path, name: str) -> logging.Logger:
    """Set up a logger that writes to both console and a log file."""
    log_dir = Path(project_dir) / 'logs'
    log_dir.mkdir(parents=True, exist_ok=True)  # Create logs/ if it doesn't exist

    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    if not logger.handlers:  # Prevent duplicate handlers when running interactively
        fmt = logging.Formatter('%(asctime)s [%(levelname)s] %(message)s', datefmt='%Y-%m-%d %H:%M:%S')
        ch  = logging.StreamHandler()                                            # Prints to console
        ch.setFormatter(fmt)
        fh  = logging.FileHandler(log_dir / f'{name}.log', encoding='utf-8')    # Writes to file
        fh.setFormatter(fmt)
        logger.addHandler(ch)
        logger.addHandler(fh)
    return logger


def ensure_dirs(project_dir: Path):
    """Create standard output directories if they don't already exist."""
    for sub in ('data/augmented', 'data/splits', 'logs', 'results'):
        (Path(project_dir) / sub).mkdir(parents=True, exist_ok=True)

## Text Cleaning & Deduplication

Synthetic reviews are cleaned with the **same normalisation** applied to real reviews, so they share an identical format and can be mixed without introducing distribution shift.

In [ ]:
def clean_text(text: str) -> str:
    """
    Lightweight text cleaning for synthetic reviews.
    Steps:
      1. Strip HTML tags   (e.g. <br>, <b>)
      2. Replace emojis with a space (emoji characters cause tokeniser issues)
      3. Collapse whitespace into single spaces
      4. Lowercase + remove URLs, emails, line breaks (via cleantext library)
    """
    text = str(text)
    text = re.sub(r'<.*?>', ' ', text)              # Step 1: strip HTML tags
    text = emoji.replace_emoji(text, replace=' ')   # Step 2: replace emoji with space
    text = re.sub(r'\s+', ' ', text)               # Step 3: collapse whitespace
    text = clean(text, lower=True, no_urls=True, no_emails=True, no_line_breaks=True)  # Step 4
    return text.strip()


def make_signature(row: pd.Series, aspects: list) -> str:
    """
    Build a compact deduplication key by joining all aspect labels.
    Used to detect if two different-worded reviews have the same label pattern.
    Example: 'stayingpower:na|texture:positive|smell:na|colour:negative|...'
    Missing aspect values are represented as 'na'.
    """
    return '|'.join(
        f"{a}:{row[a] if pd.notna(row[a]) and row[a] is not None else 'na'}"
        for a in aspects
    )

## Robust JSON Extraction

LLMs may generate files that are not valid JSON at the top level (missing brackets, trailing commas between objects). The regex approach extracts individual `{...}` objects regardless of the outer structure.

In [ ]:
def extract_json_objects(file_path, logger):
    """
    Robustly extract all JSON objects { ... } from a file,
    even if the overall structure (commas, brackets) is broken.

    Strategy: regex to find non-nested { ... } blocks.
    Limitation: won't work if review objects contain nested sub-objects.
    This schema is flat, so this is safe here.
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # Regex: match content between { and } (no nesting allowed by [^{}])
    matches = re.findall(r'\{[^{}]+\}', content, re.DOTALL)

    data = []
    for match in matches:
        try:
            obj = json.loads(match)   # Parse the individual object cleanly
            data.append(obj)
        except json.JSONDecodeError:
            # Skip objects that can't be parsed (e.g. trailing commas inside)
            continue

    if not data:
        logger.error(f'No valid JSON objects found in {file_path.name}')
        return None

    logger.info(f'Extracted {len(data)} objects from {file_path.name}')
    return data

## Main Processing Loop
Converts every `.json` file in `data/augmented/` to a matching `.csv`.

In [ ]:
import os
ML_RESEARCH = os.path.dirname(os.path.abspath(''))

def main(project_dir: str):
    project_dir, cfg = load_cfg(project_dir)
    ensure_dirs(project_dir)
    logger  = setup_logger(project_dir, 'convert_synthetic_robust')

    aspects = cfg['aspects']['names']         # e.g. ['stayingpower', 'texture', ...]
    augmented_dir = Path(project_dir) / 'data' / 'augmented'

    json_files = list(augmented_dir.glob('*.json'))  # Find all JSON files in the augmented folder
    for json_path in json_files:
        logger.info(f'Processing {json_path.name}...')

        data = extract_json_objects(json_path, logger)
        if data is None:
            continue                             # Skip files with no valid reviewobjects

        df = pd.DataFrame(data)

        # Some LLM outputs use 'review_text' instead of 'data' — normalise the column name
        if 'review_text' in df.columns:
            df = df.rename(columns={'review_text': 'data'})

        # Ensure ALL aspect columns exist (fill missing with None)
        for a in aspects:
            if a not in df.columns:
                df[a] = None

        df = df[df['data'].notna()]              # Drop rows where review text is missing

        df['text_clean'] = df['data'].apply(clean_text)      # Apply cleaning pipeline
        df['signature']  = df.apply(lambda row: make_signature(row, aspects), axis=1)  # Dedup key

        # Enforce the final column order to match the pipeline schema
        cols = ['data'] + aspects + ['text_clean', 'signature']
        df   = df[cols]

        csv_path = json_path.with_suffix('.csv')  # Same filename, .csv extension
        df.to_csv(csv_path, index=False)
        logger.info(f'Saved CSV -> {csv_path.name}')


# ── CLI Usage ──────────────────────────────────────────────────────────────────
# Usage: python convert_synthetic_robust.py --project_dir <path_to_ml-research>
if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--project_dir', required=True)
    args = parser.parse_args()
    main(args.project_dir)